# Timing versus Intensity of Breeding-Site Clean-Up Campaigns for Dengue Control in Colombo

A host–vector dengue model (SEIR humans, SEI *Aedes* mosquitoes), solved with hand-written Euler and RK4 integrators and calibrated with a hand-written Metropolis MCMC sampler to the 2026 monsoon wave of weekly reported cases in Colombo District (weeks 20–36). The calibrated model gives $R_0 = 1.96$ (95% CrI 1.83–2.08) and reproduces every observed week inside its 95% predictive band.

**Key result:** timing matters far more than intensity. A clean-up campaign halves the peak only if it starts about 8 weeks before the peak and removes at least half of new adult mosquitoes (median 51% reduction; at 60% the whole 95% interval is above 50%). Campaigns started 4 weeks or less before the peak cut it by at most 27%, even at 80% intensity, because of the 3–4-week delay between removing breeding sites and fewer human cases.

**Verification:** conservation, $\beta = 0$ limit, closed-form vs next-generation $R_0$, the analytic threshold $p_c$, measured Euler/RK4 orders (0.98, 4.03), negative-binomial check, parameter recovery from synthetic data, and Monte Carlo error falling as $N^{-1/2}$, all enforced with `assert`.

In [ ]:
import os, time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy import stats, optimize
from scipy.special import gammaln

T_START = time.perf_counter()     # used by the last cell to report the Run-all time
SEED = 42                                  # global seed, used by every random step in this notebook
rng = np.random.default_rng(SEED)
os.makedirs("figures", exist_ok=True)
print("numpy", np.__version__, "| scipy", scipy.__version__, "| pandas", pd.__version__, "| matplotlib", matplotlib.__version__)

## 1. Problem and question

Dengue is endemic in Sri Lanka, and Colombo District is consistently among the most affected districts. Cases rise sharply after the south-west monsoon, as they did again in 2026. The main community response is breeding-site clean-up, which reduces the number of new adult *Aedes* mosquitoes. Health authorities must decide when to start a campaign and how intensive to make it.

**Question:** how early and how large must a breeding-site clean-up campaign be to halve the peak weekly dengue case count in Colombo District?

The answer is reported with posterior uncertainty bands, not single runs.

## 2. Model, assumptions and equations

**State variables.** Humans are tracked as fractions of the effective susceptible population $N$: susceptible $s$, exposed $e$, infectious $i$, recovered $r$, and cumulative incidence $c$. Mosquitoes are tracked per human: susceptible $V_s$, exposed $V_e$, infectious $V_i$.

$$
\begin{aligned}
\frac{ds}{dt} &= -\beta V_i s, &
\frac{dV_s}{dt} &= \lambda(t) - \beta i V_s - \mu_v V_s,\\
\frac{de}{dt} &= \beta V_i s - \sigma_h e, &
\frac{dV_e}{dt} &= \beta i V_s - (\sigma_v + \mu_v) V_e,\\
\frac{di}{dt} &= \sigma_h e - \gamma i, &
\frac{dV_i}{dt} &= \sigma_v V_e - \mu_v V_i,\\
\frac{dr}{dt} &= \gamma i, &
\frac{dc}{dt} &= \sigma_h e .
\end{aligned}
$$

**Clean-up campaign.** A campaign that starts on day $t_c$ cuts mosquito recruitment by a fraction $p$:

$$\lambda(t) = m_0 \mu_v \left(1 - p\,\mathbf{1}[t \ge t_c]\right).$$

**Basic reproduction number** (next-generation matrix, host → vector → host):

$$R_0^2 = \frac{\beta^2 m_0 \sigma_v}{\gamma\, \mu_v (\sigma_v + \mu_v)}, \qquad R_0(p)^2 = R_0^2 (1 - p), \qquad p_c = 1 - \frac{1}{R_0^2}.$$

A campaign applied from the start stops the outbreak exactly when $p > p_c$.

**Observation model** (used from section 5): weekly reported cases $y_w \sim \text{NegBin}\big(\mu_w = \rho N \Delta c_w,\ k\big)$.

| Symbol | Meaning | Units | Value | Source |
|---|---|---|---|---|
| $\sigma_h$ | 1 / human intrinsic incubation period | 1/day | 1/5.5 | [literature, to cite] |
| $\gamma$ | 1 / human infectious period | 1/day | 1/5 | [literature, to cite] |
| $\sigma_v$ | 1 / extrinsic incubation period in the mosquito | 1/day | 1/10 | [literature, to cite] |
| $\mu_v$ | adult *Aedes* death rate (1 / lifespan) | 1/day | 1/14 | [literature, to cite] |
| $m_0$ | baseline adult mosquitoes per human | – | 2 | assumed; varied in sensitivity |
| $N$ | effective susceptible population = susceptible fraction × district population | people | 0.30 × 2,375,415 | Census 2024; fraction assumed |
| $\beta$ | transmission rate (same in both directions) | 1/day | fitted | – |
| $\rho$ | reporting fraction | – | fitted | – |
| $I_0$ | infectious people at the start of the fitting window (week 20) | people | fitted | – |
| $k$ | negative-binomial dispersion | – | 20 | assumed |

**Assumptions**
1. Closed population over one season; births, deaths and migration of humans are ignored.
2. One circulating serotype with lasting immunity within the season.
3. Homogeneous mixing across the district.
4. The same transmission rate $\beta$ applies human → mosquito and mosquito → human.
5. A campaign cuts the recruitment of new adult mosquitoes; adults already alive die off at their natural rate.
6. The susceptible fraction is fixed (it trades off exactly with $\rho$ within one wave) and is varied in the sensitivity analysis.
7. The reporting fraction $\rho$ is constant over the season.

**Concepts used:** population and epidemic models (SEIR / SEI); numerical ODE solvers (Euler, RK4, error orders); MCMC and the Metropolis algorithm; Monte Carlo simulation and input modelling.

## 3. Implementation

### 3.1 Model and solvers

All solvers and the model are written from scratch with NumPy. The time step is $\Delta t = 0.5$ day.

In [ ]:
N_POP = 2_375_415        # Colombo District population, Census of Population and Housing 2024 (final report)
SUSC_FRAC = 0.30         # ASSUMED share of the population susceptible to the circulating serotype (varied in sensitivity)
N = SUSC_FRAC * N_POP    # effective susceptible population [people]

FIXED = dict(
    sigma_h = 1/5.5,     # 1 / human intrinsic incubation period   [1/day]
    gamma   = 1/5.0,     # 1 / human infectious period             [1/day]
    sigma_v = 1/10.0,    # 1 / extrinsic incubation in the mosquito [1/day]
    mu_v    = 1/14.0,    # 1 / adult Aedes lifespan                [1/day]
    m0      = 2.0,       # baseline adult mosquitoes per human     [-]
)
DT = 0.5                 # RK4 time step [day]
# state vector y = [s, e, i, r, c, Vs, Ve, Vi]
#   s, e, i, r : fractions of N (sum to 1);  c : cumulative incidence (fraction of N)
#   Vs, Ve, Vi : mosquitoes per human

In [ ]:
def rhs(t, y, beta, lam, P=FIXED):
    """Host-vector SEIR-SEI derivatives. lam = mosquito recruitment rate [mosquitoes per human per day]."""
    s, e, i, r, c, Vs, Ve, Vi = y
    sh, g, sv, mv = P["sigma_h"], P["gamma"], P["sigma_v"], P["mu_v"]
    inf_h = beta * Vi * s            # new human infections   [1/day]
    inf_v = beta * i * Vs            # new mosquito infections [1/day]
    return np.array([
        -inf_h,
        inf_h - sh * e,
        sh * e - g * i,
        g * i,
        sh * e,                      # cumulative incidence -> weekly cases
        lam - inf_v - mv * Vs,
        inf_v - (sv + mv) * Ve,
        sv * Ve - mv * Vi,
    ])

Both solvers are fixed-step. Explicit Euler has global error $O(\Delta t)$; classical RK4 has global error $O(\Delta t^4)$ for smooth right-hand sides.

In [ ]:
def euler(f, y0, t0, t1, dt):
    """Explicit Euler, fixed step. Returns times and states (rows)."""
    n = int(round((t1 - t0) / dt))
    ts = t0 + dt * np.arange(n + 1)
    ys = np.empty((n + 1, len(y0))); ys[0] = y0
    for k in range(n):
        ys[k + 1] = ys[k] + dt * f(ts[k], ys[k])
    return ts, ys

def rk4(f, y0, t0, t1, dt):
    """Classical fourth-order Runge-Kutta, fixed step. Returns times and states (rows)."""
    n = int(round((t1 - t0) / dt))
    ts = t0 + dt * np.arange(n + 1)
    ys = np.empty((n + 1, len(y0))); ys[0] = y0
    for k in range(n):
        t, y = ts[k], ys[k]
        k1 = f(t, y)
        k2 = f(t + dt/2, y + dt/2 * k1)
        k3 = f(t + dt/2, y + dt/2 * k2)
        k4 = f(t + dt, y + dt * k3)
        ys[k + 1] = y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
    return ts, ys

The campaign makes $\lambda(t)$ jump at $t_c$. If a solver step straddled the jump, RK4 would lose its fourth-order accuracy. `simulate` therefore integrates in two segments that meet exactly at $t_c$, which must lie on the time grid.

In [ ]:
def initial_state(I0, m=None, P=FIXED):
    """I0 infectious people at t = 0; mosquitoes start uninfected at density m (default m0)."""
    m = P["m0"] if m is None else m
    i0 = I0 / N
    return np.array([1 - i0, 0.0, i0, 0.0, 0.0, m, 0.0, 0.0])

def simulate(beta, I0, n_weeks, p=0.0, t_c=np.inf, dt=DT, solver=rk4, y0=None, P=FIXED):
    """Run the model for n_weeks. A clean-up campaign cuts mosquito recruitment by fraction p from day t_c.
    Integrated in two segments split exactly at t_c, so no solver step straddles the jump in recruitment."""
    T = 7 * n_weeks
    y0 = initial_state(I0, P=P) if y0 is None else y0
    lam0 = P["m0"] * P["mu_v"]
    t_c = min(max(t_c, 0.0), T)
    assert abs(t_c / dt - round(t_c / dt)) < 1e-9, "t_c must lie on the time grid"
    segs = [(0.0, t_c, lam0), (t_c, T, lam0 * (1 - p))]
    ts_all, ys_all = [np.array([0.0])], [y0[None, :]]
    y = y0
    for a, b, lam in segs:
        if b - a < 1e-12:
            continue
        ts, ys = solver(lambda t, yy: rhs(t, yy, beta, lam, P), y, a, b, dt)
        ts_all.append(ts[1:]); ys_all.append(ys[1:]); y = ys[-1]
    return np.concatenate(ts_all), np.vstack(ys_all)

def weekly_cases(ts, ys, rho):
    """Expected reported cases per week = rho * N * (increase in cumulative incidence over the week)."""
    week_idx = [np.argmin(abs(ts - 7 * w)) for w in range(int(round(ts[-1] / 7)) + 1)]
    c_week = ys[week_idx, 4]
    return rho * N * np.diff(c_week)

def R0(beta, p=0.0, P=FIXED):
    """Basic reproduction number (host -> vector -> host, square root of the two-generation product)."""
    m = P["m0"] * (1 - p)
    return np.sqrt(beta**2 * m * P["sigma_v"] / (P["gamma"] * P["mu_v"] * (P["sigma_v"] + P["mu_v"])))

def p_crit(beta, P=FIXED):
    """Smallest recruitment cut that stops an outbreak: R0(p)^2 = R0^2 (1 - p) = 1."""
    return 1 - 1 / R0(beta, P=P)**2

BETA_TEST, I0_TEST = 0.35, 50
ts, ys = simulate(BETA_TEST, I0_TEST, n_weeks=25)
wk = weekly_cases(ts, ys, rho=0.1)
print(f"R0 = {R0(BETA_TEST):.3f}, p_c = {p_crit(BETA_TEST):.3f}")
print(f"peak weekly cases (rho = 0.1) = {wk.max():.0f} in week {wk.argmax() + 1}")
t0 = time.perf_counter()
for _ in range(20): simulate(BETA_TEST, I0_TEST, n_weeks=25)
print(f"one 25-week RK4 solve: {(time.perf_counter() - t0) / 20 * 1000:.1f} ms")

### 3.2 Data

Weekly notified dengue cases for Colombo District, 2026, from the Epidemiology Unit's *Weekly Epidemiological Report* (Table 1), as extracted in the `denguedatahub` package; see `data/README.md`. Five weeks with extraction errors are marked missing and are skipped by the likelihood, not filled in.

**Fitting window: weeks 20–36 (11 May – 6 Sep 2026).** Weeks 16–19 are a flat pre-monsoon period (about 250–330 cases per week). A model with one constant transmission rate cannot stay flat and then rise steeply, so including them biases the fit: with a start at week 16, only 78% of observed weeks fell inside the 95% predictive band at the best-fit parameters, against 93% for week 19 and 100% for weeks 20 and 21. The window therefore covers the monsoon wave itself.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/KavinduG13/dengue-cleanup-simulator/main/data/colombo_weekly_dengue_2026.csv"
df = pd.read_csv(DATA_URL, parse_dates=["week_start", "week_end"])
FIT_START, FIT_END = 20, 36                        # fitting window (epi weeks): the monsoon wave, see the Markdown cell above
fit = df[(df.epi_week >= FIT_START) & (df.epi_week <= FIT_END)].reset_index(drop=True)
N_WEEKS = len(fit)
y_obs = fit.cases.to_numpy(dtype=float)            # NaN where the week is missing
obs_mask = ~np.isnan(y_obs)
print(f"Fitting window: weeks {FIT_START}-{FIT_END} ({N_WEEKS} weeks, {obs_mask.sum()} observed, "
      f"{(~obs_mask).sum()} missing: {fit.epi_week[~obs_mask].tolist()})")

fig, ax = plt.subplots(figsize=(8, 3.8))
ok = df[df.status == "ok"]
ax.plot(ok.week_start, ok.cases, "o-", ms=4, label="reported cases")
for t in df.week_start[df.status == "missing"]:
    ax.axvline(t, color="0.8", lw=5, alpha=0.6)
ax.axvspan(pd.Timestamp("2026-06-15"), pd.Timestamp("2026-06-21"), color="orange", alpha=0.3, label="control week 15-21 Jun")
ax.axvspan(fit.week_start.iloc[0], fit.week_end.iloc[-1], color="tab:blue", alpha=0.06, label="fitting window")
ax.set_ylabel("cases per week"); ax.set_title("Colombo District, 2026 (grey = missing week)"); ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 3.3 Starting state

At the start of the window the epidemic is already under way. Starting with no exposed people and no infected mosquitoes would create an artificial lag of about two weeks. Instead, the infected compartments $(e, i, V_e, V_i)$ start on the dominant eigenvector of the linearised system $\dot x = (F - V)x$, scaled so that $I_0$ people are infectious: this is the mix of compartments that a growing epidemic settles into.

In [ ]:
def infected_jacobian(beta, P=FIXED):
    """Linearised dynamics of (e, i, Ve, Vi) near the disease-free state: dx/dt = (F - V) x."""
    sh, g, sv, mv, m = P["sigma_h"], P["gamma"], P["sigma_v"], P["mu_v"], P["m0"]
    return np.array([[-sh,  0.0,     0.0,          beta],
                     [ sh, -g,       0.0,          0.0 ],
                     [0.0,  beta*m, -(sv + mv),    0.0 ],
                     [0.0,  0.0,     sv,          -mv  ]])

def initial_state_growth(beta, I0, P=FIXED):
    """Start on the dominant eigenvector, so infected humans and mosquitoes are already in the
    proportions of a growing (or shrinking) epidemic, with I0 infectious people. This removes the
    artificial lag of starting with no exposed people and no infected mosquitoes."""
    w, v = np.linalg.eig(infected_jacobian(beta, P))
    vec = np.abs(np.real(v[:, np.argmax(np.real(w))]))
    e0, i0, Ve0, Vi0 = vec / vec[1] * (I0 / N)
    return np.array([1 - e0 - i0, e0, i0, 0.0, 0.0, P["m0"] - Ve0 - Vi0, Ve0, Vi0])

def expected_cases(theta, n_weeks=N_WEEKS, p=0.0, t_c=np.inf, dt=DT, P=FIXED):
    """Expected weekly reported cases for theta = (beta, rho, I0)."""
    beta, rho, I0 = theta
    ts, ys = simulate(beta, I0, n_weeks, p=p, t_c=t_c, dt=dt, y0=initial_state_growth(beta, I0, P), P=P)
    return weekly_cases(ts, ys, rho)

### 3.4 Observation model and Metropolis sampler

Reported cases in week $w$ are negative binomial with mean $\mu_w = \rho N \Delta c_w$ and dispersion $k = 20$, so $\text{Var}(y_w) = \mu_w + \mu_w^2/k$.

The sampler works in $\phi = (\log\beta,\ \text{logit}\,\rho,\ \log I_0)$, so proposals never leave the valid range. The priors are uniform on the natural scale, $\beta \in [0.05, 1.5]$, $\rho \in [0.001, 1]$, $I_0 \in [1, 10^5]$, so the log-Jacobian of the transform, $\log\beta + \log\rho + \log(1-\rho) + \log I_0$, is added to the log-posterior.

Procedure: a Nelder–Mead search finds the posterior mode; a 1,000–1,500-iteration pilot chain estimates the posterior covariance; the main chains then use Gaussian random-walk proposals with covariance $2.38^2/d$ times that estimate ($d = 3$). The proposal is fixed during the main runs, so they are ordinary Metropolis chains. Convergence is checked with the Gelman–Rubin $\hat R$ across two chains started from over-dispersed points, and with the effective sample size (ESS).

In [ ]:
K_NB = 20.0   # dispersion: Var = mu + mu^2 / k

def nb_logpmf(y, mu, k=K_NB):
    """log P(Y = y) for a negative binomial with mean mu and dispersion k."""
    mu = np.maximum(mu, 1e-9)
    return (gammaln(y + k) - gammaln(k) - gammaln(y + 1)
            + k * np.log(k / (k + mu)) + y * np.log(mu / (k + mu)))

In [ ]:
BOUNDS = {"beta": (0.05, 1.5), "rho": (0.001, 1.0), "I0": (1.0, 1e5)}   # uniform priors on the natural scale

def to_theta(phi):
    """phi = (log beta, logit rho, log I0) -> theta = (beta, rho, I0)."""
    return np.array([np.exp(phi[0]), 1 / (1 + np.exp(-phi[1])), np.exp(phi[2])])

def to_phi(theta):
    b, r, i = theta
    return np.array([np.log(b), np.log(r / (1 - r)), np.log(i)])

def log_post(phi, y=None, mask=None):
    """Log posterior in phi-space. The priors are uniform on (beta, rho, I0), so the Jacobian of the
    transform, log beta + log rho + log(1 - rho) + log I0, is added."""
    y = y_obs if y is None else y
    mask = obs_mask if mask is None else mask
    b, r, i = theta = to_theta(phi)
    for val, (lo, hi) in zip(theta, BOUNDS.values()):
        if not lo <= val <= hi:
            return -np.inf
    mu = expected_cases(theta)
    log_jac = np.log(b) + np.log(r) + np.log(1 - r) + np.log(i)
    return nb_logpmf(y[mask], mu[mask]).sum() + log_jac

def metropolis(logp, phi0, cov, n_iter, rng):
    """Random-walk Metropolis with Gaussian proposals N(phi, cov). Returns the chain and acceptance rate."""
    L = np.linalg.cholesky(cov)
    chain = np.empty((n_iter, len(phi0)))
    phi, lp, acc = np.array(phi0, float), logp(phi0), 0
    for t in range(n_iter):
        prop = phi + L @ rng.standard_normal(len(phi))
        lp_prop = logp(prop)
        if np.log(rng.uniform()) < lp_prop - lp:
            phi, lp, acc = prop, lp_prop, acc + 1
        chain[t] = phi
    return chain, acc / n_iter

def gelman_rubin(chains):
    """Potential scale reduction factor R-hat for each parameter; chains has shape (m, n, d)."""
    m, n, _ = chains.shape
    W = chains.var(axis=1, ddof=1).mean(axis=0)
    B = n * chains.mean(axis=1).var(axis=0, ddof=1)
    return np.sqrt(((n - 1) / n * W + B / n) / W)

def ess(x):
    """Effective sample size from the initial positive sequence of autocorrelations."""
    x = x - x.mean(); n = len(x)
    ac = np.correlate(x, x, "full")[n - 1:] / (x.var() * np.arange(n, 0, -1))
    s = 0.0
    for k in range(1, n - 1, 2):
        pair = ac[k] + ac[k + 1]
        if pair < 0:
            break
        s += pair
    return n / (1 + 2 * s)

def calibrate(logp, phi_start, n_pilot, n_main, burn, rng, n_chains=2, label=""):
    """MAP start -> pilot run to learn the proposal covariance -> n_chains main runs."""
    t0 = time.perf_counter()
    res = optimize.minimize(lambda p: -logp(p), phi_start, method="Nelder-Mead",
                            options={"xatol": 1e-4, "fatol": 1e-4, "maxiter": 2000})
    pilot, acc_p = metropolis(logp, res.x, np.diag([0.02, 0.05, 0.05])**2, n_pilot, rng)
    cov = np.cov(pilot[n_pilot // 2:].T) * 2.38**2 / 3 + 1e-10 * np.eye(3)
    chains, accs = [], []
    for c in range(n_chains):
        start = res.x + rng.multivariate_normal(np.zeros(3), 4 * cov)   # over-dispersed starts
        ch, a = metropolis(logp, start, cov, n_main, rng)
        chains.append(ch[burn:]); accs.append(a)
    chains = np.array(chains)
    rhat = gelman_rubin(chains)
    ess_ = [ess(chains[:, :, j].ravel()) for j in range(3)]
    print(f"{label}: pilot acceptance {acc_p:.2f}; main acceptance {', '.join(f'{a:.2f}' for a in accs)}; "
          f"R-hat {np.round(rhat, 3)}; ESS {np.round(ess_).astype(int)}; {time.perf_counter() - t0:.0f} s")
    return chains, accs, rhat

def summarise(samples, names=("beta", "rho", "I0")):
    q = np.percentile(samples, [2.5, 50, 97.5], axis=0)
    return pd.DataFrame(q.T, index=names, columns=["2.5%", "median", "97.5%"])

## 4. Verification

Each check ends in an `assert`, so a failure stops *Run all*.

**V1. Conservation.** Without a campaign, $s+e+i+r = 1$ and $V_s+V_e+V_i = m_0$ for all time. With a campaign from $t_c$, the mosquito total obeys $\dot V = m_0\mu_v(1-p) - \mu_v V$, so $V(t) = m_0\left[1 - p\left(1 - e^{-\mu_v (t - t_c)}\right)\right]$.

In [ ]:
ts, ys = simulate(BETA_TEST, I0_TEST, n_weeks=25)
err_h = np.max(np.abs(ys[:, :4].sum(axis=1) - 1))
err_v = np.max(np.abs(ys[:, 5:].sum(axis=1) - FIXED["m0"]))
print(f"max |s+e+i+r - 1| = {err_h:.2e},  max |Vs+Ve+Vi - m0| = {err_v:.2e}")
assert err_h < 1e-10 and err_v < 1e-10
print("PASS: humans and mosquitoes conserved (no campaign)")

# with a campaign the mosquito total relaxes to m0 (1 - p)
ts, ys = simulate(BETA_TEST, I0_TEST, n_weeks=40, p=0.4, t_c=35.0)
Vtot = ys[:, 5:].sum(axis=1)
expected = FIXED["m0"] * (1 - 0.4 * (1 - np.exp(-FIXED["mu_v"] * np.clip(ts - 35.0, 0, None))))
err = np.max(np.abs(Vtot - expected))
print(f"max |V_total - analytic m0(1 - p(1 - e^(-mu_v (t - t_c))))| = {err:.2e}")
assert err < 1e-8
print("PASS: mosquito total follows the analytic relaxation after the campaign starts")

**V2. Limiting case.** With $\beta = 0$ there is no transmission: cumulative incidence and infected mosquitoes stay at zero, and infectious humans decay as $i_0 e^{-\gamma t}$. The tolerance allows for RK4's own error on this exponential (about $10^{-7}$ relative at $\Delta t = 0.5$).

In [ ]:
ts, ys = simulate(0.0, I0_TEST, n_weeks=25)
assert np.max(ys[:, 4]) < 1e-15 and np.max(ys[:, 6:]) < 1e-15
i_exact = (I0_TEST / N) * np.exp(-FIXED["gamma"] * ts)
rel = np.max(np.abs(ys[:, 2] - i_exact)) / (I0_TEST / N)
print(f"max relative error vs i0 exp(-gamma t) = {rel:.1e}")
assert rel < 1e-5
print("PASS: beta = 0 gives no new infections, and i(t) decays as i0 exp(-gamma t)")

**V3. Independent $R_0$ check.** $R_0$ is recomputed numerically as the spectral radius of the next-generation matrix $K = F V^{-1}$ for the infected compartments $(e, i, V_e, V_i)$, and compared with the closed form.

In [ ]:
def R0_ngm(beta, P=FIXED):
    sh, g, sv, mv, m = P["sigma_h"], P["gamma"], P["sigma_v"], P["mu_v"], P["m0"]
    # infected compartments x = (e, i, Ve, Vi) at the disease-free state (s = 1, Vs = m0)
    F = np.array([[0, 0,      0, beta],
                  [0, 0,      0, 0   ],
                  [0, beta*m, 0, 0   ],
                  [0, 0,      0, 0   ]], dtype=float)
    V = np.array([[ sh, 0, 0,       0 ],
                  [-sh, g, 0,       0 ],
                  [ 0,  0, sv + mv, 0 ],
                  [ 0,  0, -sv,     mv]], dtype=float)
    K = F @ np.linalg.inv(V)
    return np.max(np.abs(np.linalg.eigvals(K)))

for b in [0.1, 0.2, 0.35, 0.5]:
    a, n_ = R0(b), R0_ngm(b)
    print(f"beta = {b:.2f}: closed form R0 = {a:.6f}, next-generation matrix = {n_:.6f}")
    assert abs(a - n_) < 1e-8
print("PASS: closed-form R0 equals the spectral radius of the next-generation matrix")

**V4. Analytic threshold.** Mosquitoes start at the reduced equilibrium $m_0(1-p)$ and the campaign runs from $t = 0$. Just below $p_c$ the late-time growth rate of infectious humans must be positive; just above $p_c$ it must be negative.

In [ ]:
def growth_rate(beta, p, days=600):
    """Late-time exponential growth rate of infectious humans, starting at the reduced mosquito equilibrium."""
    m = FIXED["m0"] * (1 - p)
    y0 = initial_state(1e-3, m=m)            # tiny seed keeps s close to 1 (linear regime)
    ts, ys = simulate(beta, 1e-3, n_weeks=days // 7, p=p, t_c=0.0, y0=y0)
    late = ts > ts[-1] - 100
    return np.polyfit(ts[late], np.log(ys[late, 2]), 1)[0]

pc = p_crit(BETA_TEST)
r_below, r_above = growth_rate(BETA_TEST, pc - 0.03), growth_rate(BETA_TEST, pc + 0.03)
print(f"p_c = {pc:.4f}: growth rate at p_c - 0.03 = {r_below:+.4f}/day, at p_c + 0.03 = {r_above:+.4f}/day")
assert r_below > 0 > r_above
print("PASS: the outbreak grows just below p_c and dies out just above it")

**V5. Time-step convergence.** Cumulative incidence at week 25 is compared with a reference RK4 solution at $\Delta t = 1/128$ day. The fitted log–log slopes must match the theoretical orders: 1 for Euler and 4 for RK4.

In [ ]:
def c_final(solver, dt):
    ts, ys = simulate(BETA_TEST, I0_TEST, n_weeks=25, dt=dt, solver=solver)
    return ys[-1, 4]

ref = c_final(rk4, 1/128)
dts = np.array([1.0, 0.5, 0.25, 0.125, 0.0625])
err_e = np.array([abs(c_final(euler, d) - ref) for d in dts])
err_r = np.array([abs(c_final(rk4, d) - ref) for d in dts])
slope_e = np.polyfit(np.log(dts), np.log(err_e), 1)[0]
slope_r = np.polyfit(np.log(dts), np.log(err_r), 1)[0]
print("dt      Euler error   RK4 error")
for d, a, b in zip(dts, err_e, err_r): print(f"{d:<7} {a:.3e}     {b:.3e}")
print(f"fitted order: Euler {slope_e:.2f} (expected 1), RK4 {slope_r:.2f} (expected 4)")
assert 0.8 < slope_e < 1.2 and 3.5 < slope_r < 4.5
print("PASS: measured orders match theory")

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(dts, err_e, "o-", label=f"Euler (slope {slope_e:.2f})")
ax.loglog(dts, err_r, "s-", label=f"RK4 (slope {slope_r:.2f})")
ax.loglog(dts, err_e[0] * dts / dts[0], "k--", lw=0.8, label="slope 1")
ax.loglog(dts, err_r[0] * (dts / dts[0])**4, "k:", lw=0.8, label="slope 4")
ax.set_xlabel("time step dt [day]"); ax.set_ylabel("|error| in cumulative incidence at week 25")
ax.set_title("Time-step convergence"); ax.legend(); fig.tight_layout()
fig.savefig("figures/convergence.png", dpi=150)
plt.show()

**V6. Calibration building blocks.** The eigenvector starting state must still conserve humans and mosquitoes, and the hand-written negative-binomial log-probability must match `scipy.stats.nbinom`. That check catches the common mistake of mixing up scipy's $(n, p)$ parameterisation with mean and dispersion.

In [ ]:
ts, ys = simulate(0.3, 2000, N_WEEKS, y0=initial_state_growth(0.3, 2000))
assert np.max(np.abs(ys[:, :4].sum(axis=1) - 1)) < 1e-10 and np.max(np.abs(ys[:, 5:].sum(axis=1) - FIXED["m0"])) < 1e-10
print("PASS: the eigenvector start still conserves humans and mosquitoes")

yy, mm = np.array([0, 3, 250, 1600]), np.array([5.0, 3.5, 300.0, 1400.0])
assert np.allclose(nb_logpmf(yy, mm), stats.nbinom.logpmf(yy, n=K_NB, p=K_NB / (K_NB + mm)), atol=1e-10)
print("PASS: own negative-binomial log-pmf matches scipy.stats.nbinom")

**V7. Parameter recovery.** Synthetic weekly cases are generated from known parameters $\theta_{\text{true}} = (0.22,\ 0.02,\ 5000)$, with the same weeks, the same missing weeks and negative-binomial noise. The sampler must recover all three true values inside their 95% credible intervals, with $\hat R < 1.1$. While building the notebook, this check was repeated on five independent synthetic datasets and passed every time.

In [ ]:
rng_syn = np.random.default_rng(SEED + 1)
THETA_TRUE = np.array([0.22, 0.02, 5000.0])
mu_true = expected_cases(THETA_TRUE)
y_syn = rng_syn.negative_binomial(K_NB, K_NB / (K_NB + mu_true)).astype(float)
y_syn[~obs_mask] = np.nan                           # same missing weeks as the real data
logp_syn = lambda phi: log_post(phi, y=y_syn)

chains_syn, acc_syn, rhat_syn = calibrate(logp_syn, to_phi([0.4, 0.1, 1000.0]), 1000, 3000, 750, rng, label="synthetic")
post_syn = np.array([to_theta(p) for p in chains_syn.reshape(-1, 3)])
tab = summarise(post_syn); tab["true"] = THETA_TRUE; print(tab)
inside = (tab["2.5%"] <= tab["true"]) & (tab["true"] <= tab["97.5%"])
assert inside.all(), "a true value lies outside its 95% credible interval"
assert (rhat_syn < 1.1).all()
print("PASS: all three true parameters lie inside their 95% credible intervals, R-hat < 1.1")

## 5. Experiments and results

### 5.1 Calibration to the Colombo 2026 wave

In [ ]:
chains, accs, rhat = calibrate(log_post, to_phi([0.4, 0.1, 1000.0]), 1500, 6000, 1500, rng, label="Colombo")
assert (rhat < 1.1).all(), "chains have not converged"
assert all(0.15 < a < 0.5 for a in accs), "acceptance rate outside the usual range"
post = np.array([to_theta(p) for p in chains.reshape(-1, 3)])
R0_post = R0(post[:, 0]); pc_post = p_crit(post[:, 0])
tab = summarise(np.column_stack([post, R0_post, pc_post]), names=("beta", "rho", "I0", "R0", "p_c"))
print(tab.round(4))
THETA_MED = np.median(post, axis=0)

fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for j, name in enumerate(["log beta", "logit rho", "log I0"]):
    for c in range(chains.shape[0]):
        axes[j].plot(chains[c, :, j], lw=0.4)
    axes[j].set_ylabel(name)
axes[-1].set_xlabel("iteration after burn-in"); fig.suptitle("Trace plots (two chains)"); fig.tight_layout()
fig.savefig("figures/traces.png", dpi=150)
plt.show()

Posterior predictive check: 400 posterior draws are simulated, giving the 95% band of the model mean, and negative-binomial noise is added to give the 95% predictive band. A well-calibrated model should contain roughly 95% of the observed weeks in the predictive band.

In [ ]:
idx = rng.choice(len(post), 400, replace=False)
mu_draws = np.array([expected_cases(post[k]) for k in idx])
y_pred = rng.negative_binomial(K_NB, K_NB / (K_NB + mu_draws))
mean_lo, mean_med, mean_hi = np.percentile(mu_draws, [2.5, 50, 97.5], axis=0)
pred_lo, pred_hi = np.percentile(y_pred, [2.5, 97.5], axis=0)
coverage = np.mean((y_obs[obs_mask] >= pred_lo[obs_mask]) & (y_obs[obs_mask] <= pred_hi[obs_mask]))
print(f"Observed weeks inside the 95% posterior predictive band: {coverage:.0%}")

fig, ax = plt.subplots(figsize=(8, 4))
wk = fit.epi_week
ax.fill_between(wk, pred_lo, pred_hi, color="tab:blue", alpha=0.15, label="95% predictive band")
ax.fill_between(wk, mean_lo, mean_hi, color="tab:blue", alpha=0.35, label="95% band of model mean")
ax.plot(wk, mean_med, color="tab:blue", label="posterior median")
ax.plot(wk[obs_mask], y_obs[obs_mask], "ko", ms=4, label="reported cases")
ax.plot(wk[~obs_mask], fit.cases_raw[~obs_mask], "x", color="0.5", label="missing week (not fitted)")
ax.set_xlabel("epidemiological week, 2026"); ax.set_ylabel("cases per week")
ax.set_title("Calibrated model vs Colombo data"); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig("figures/fit.png", dpi=150)
plt.show()

### 5.2 Campaign scenarios: timing × intensity

A campaign cuts the recruitment of new adult mosquitoes by a fraction $p$ from a start week until the end of the season. Start weeks are measured relative to each posterior draw's own baseline peak: 8, 6, 4, 2 or 0 weeks before it. Eight weeks before the peak is the start of the fitted window (week 20), so it is the earliest start the calibrated model can assess. Each of 200 posterior draws is run under every campaign; the peak and total reduction are computed per draw, so the intervals carry parameter uncertainty. The model is run for 30 weeks from week 20, so a peak delayed past week 36 is still counted.

In [ ]:
HORIZON = 30                                     # weeks simulated from week 20 (to mid-December), so delayed peaks are counted
P_GRID = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
OFFSETS = np.array([8, 6, 4, 2, 0])              # campaign start, in weeks before the baseline peak
N_POST = 200

base_med = expected_cases(THETA_MED, n_weeks=HORIZON)
PEAK_IDX = int(np.argmax(base_med))              # weeks after the start of week 20
PEAK_WEEK = FIT_START + PEAK_IDX
print(f"Baseline (posterior median) peak: {base_med.max():.0f} reported cases in epi week {PEAK_WEEK}")
assert PEAK_IDX - OFFSETS.max() >= 0, "earliest campaign would start before the simulation"

def campaign_effect(theta, p, offset_weeks, P=FIXED):
    """Peak and total reported cases without and with a campaign starting offset_weeks before the baseline peak."""
    base = expected_cases(theta, n_weeks=HORIZON, P=P)
    t_c = 7.0 * (int(np.argmax(base)) - offset_weeks)
    camp = expected_cases(theta, n_weeks=HORIZON, p=p, t_c=max(t_c, 0.0), P=P)
    return base.max(), camp.max(), base.sum(), camp.sum()

In [ ]:
t0 = time.perf_counter()
draw = rng.choice(len(post), N_POST, replace=False)
peak_red = np.empty((len(P_GRID), len(OFFSETS), N_POST))
total_red = np.empty_like(peak_red)
for n, k in enumerate(draw):
    theta = post[k]
    base = expected_cases(theta, n_weeks=HORIZON)
    pk = int(np.argmax(base))
    for a, p in enumerate(P_GRID):
        for b, off in enumerate(OFFSETS):
            camp = expected_cases(theta, n_weeks=HORIZON, p=p, t_c=7.0 * max(pk - off, 0))
            peak_red[a, b, n] = 1 - camp.max() / base.max()
            total_red[a, b, n] = 1 - camp.sum() / base.sum()
print(f"{N_POST} posterior draws x {len(P_GRID) * len(OFFSETS)} campaigns in {time.perf_counter() - t0:.0f} s")

q_lo, q_med, q_hi = np.percentile(100 * peak_red, [2.5, 50, 97.5], axis=2)
t_lo, t_med, t_hi = np.percentile(100 * total_red, [2.5, 50, 97.5], axis=2)
rows = []
for a, p in enumerate(P_GRID):
    for b, off in enumerate(OFFSETS):
        rows.append({"reduction p": p, "weeks before peak": off,
                     "peak reduction % (median)": round(q_med[a, b], 1),
                     "peak 95% interval": f"{q_lo[a, b]:.1f} – {q_hi[a, b]:.1f}",
                     "total-case reduction % (median)": round(t_med[a, b], 1),
                     "total 95% interval": f"{t_lo[a, b]:.1f} – {t_hi[a, b]:.1f}"})
scen = pd.DataFrame(rows)
print(scen.to_string(index=False))

### 5.3 Answer to the question

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.2))
im = ax.imshow(q_med, origin="lower", cmap="Blues", vmin=0, vmax=100, aspect="auto")
for a in range(len(P_GRID)):
    for b in range(len(OFFSETS)):
        strong = q_lo[a, b] >= 50
        ax.text(b, a, f"{q_med[a, b]:.0f}%\n[{q_lo[a, b]:.0f}, {q_hi[a, b]:.0f}]", ha="center", va="center", fontsize=8,
                color="white" if q_med[a, b] > 55 else "black", fontweight="bold" if strong else "normal")
        if q_med[a, b] >= 50:
            ax.add_patch(plt.Rectangle((b - 0.5, a - 0.5), 1, 1, fill=False, lw=2.2 if strong else 1.2,
                                       ec="darkorange", ls="-" if strong else "--"))
ax.set_xticks(range(len(OFFSETS)), [f"{o}" for o in OFFSETS])
ax.set_yticks(range(len(P_GRID)), [f"{p:.0%}" for p in P_GRID])
ax.set_xlabel("campaign start: weeks before the expected peak")
ax.set_ylabel("cut in new adult mosquitoes, p")
ax.set_title("Median reduction in peak weekly cases [95% interval]\n"
             "dashed outline: median ≥ 50%; solid outline + bold: whole 95% interval ≥ 50%", fontsize=10)
fig.colorbar(im, ax=ax, label="peak reduction (%)"); fig.tight_layout()
fig.savefig("figures/heatmap.png", dpi=150); plt.show()

In [ ]:
answer = []
for b, off in enumerate(OFFSETS):
    med_ok = [P_GRID[a] for a in range(len(P_GRID)) if q_med[a, b] >= 50]
    all_ok = [P_GRID[a] for a in range(len(P_GRID)) if q_lo[a, b] >= 50]
    answer.append({"weeks before peak": off,
                   "smallest p with median >= 50%": f"{min(med_ok):.0%}" if med_ok else "none up to 80%",
                   "smallest p with 95% interval >= 50%": f"{min(all_ok):.0%}" if all_ok else "none up to 80%"})
answer = pd.DataFrame(answer)
print(answer.to_string(index=False))

### 5.4 Monte Carlo error of the scenario estimates

The scenario results are Monte Carlo averages over posterior draws. For one campaign (40%, 4 weeks before the peak), the mean peak reduction is recomputed from repeated random sub-samples of $N$ draws. Its standard error must fall as $N^{-1/2}$, a slope of $-1/2$ on a log–log plot. At $N = 200$ it is about 0.1 percentage points, far smaller than the differences between grid cells.

In [ ]:
CELL_P, CELL_OFF = 0.4, 4
pool = np.array([1 - b_c[1] / b_c[0] for b_c in
                 (campaign_effect(post[k], CELL_P, CELL_OFF) for k in rng.choice(len(post), 1000, replace=False))])
ns = np.array([25, 50, 100, 200, 400])
se = np.array([np.std([rng.choice(pool, n).mean() for _ in range(400)], ddof=1) for n in ns])
slope_mc = np.polyfit(np.log(ns), np.log(se), 1)[0]
print(f"Monte Carlo standard error of the mean peak reduction (p = {CELL_P}, {CELL_OFF} weeks before peak):")
for n, s in zip(ns, se):
    print(f"  N = {n:<4d} SE = {100 * s:.3f} percentage points")
print(f"fitted slope {slope_mc:.3f} (theory -0.5)")
assert -0.6 < slope_mc < -0.4
print("PASS: Monte Carlo error falls as N^(-1/2)")
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.loglog(ns, 100 * se, "o-", label=f"measured (slope {slope_mc:.2f})")
ax.loglog(ns, 100 * se[0] * np.sqrt(ns[0] / ns), "k--", lw=0.8, label="slope -1/2")
ax.set_xlabel("number of posterior draws N"); ax.set_ylabel("standard error (percentage points)")
ax.set_title("Monte Carlo error of the mean peak reduction"); ax.legend(); fig.tight_layout()
fig.savefig("figures/mc_error.png", dpi=150); plt.show()

### 5.5 Sensitivity to the fixed inputs

Each fixed input is changed by ±25%. The model is **refitted to the data each time** (posterior mode by Nelder–Mead), because a perturbed model that no longer matches the observed wave would say nothing useful. The output is the smallest cut $p$ that halves the peak with a campaign starting 8 weeks before it.

In [ ]:
def map_fit(P, k, phi0):
    """Posterior mode for fixed biology P and dispersion k (Nelder-Mead)."""
    def nlp(phi):
        b, r, i = th = to_theta(phi)
        if not all(lo <= v <= hi for v, (lo, hi) in zip(th, BOUNDS.values())):
            return np.inf
        mu = expected_cases(th, P=P)
        return -(nb_logpmf(y_obs[obs_mask], mu[obs_mask], k).sum() + np.log(b * r * (1 - r) * i))
    res = optimize.minimize(nlp, phi0, method="Nelder-Mead", options={"xatol": 1e-4, "fatol": 1e-4, "maxiter": 1500})
    return to_theta(res.x)

def min_p_to_halve(theta, offset, P=FIXED):
    """Smallest p (to 0.5 percentage points) that halves the peak, by bisection; NaN if even p = 0.95 cannot."""
    lo, hi = 0.0, 0.95
    b_pk, c_pk, _, _ = campaign_effect(theta, hi, offset, P)
    if 1 - c_pk / b_pk < 0.5:
        return np.nan
    while hi - lo > 0.005:
        mid = (lo + hi) / 2
        b_pk, c_pk, _, _ = campaign_effect(theta, mid, offset, P)
        lo, hi = (mid, hi) if 1 - c_pk / b_pk < 0.5 else (lo, mid)
    return hi

SENS_P, SENS_OFF = 0.5, 8                        # reference campaign: the smallest grid cell that halves the peak

def sensitivity_outputs(P, k):
    theta = map_fit(P, k, to_phi(THETA_MED))
    b_pk, c_pk, _, _ = campaign_effect(theta, SENS_P, SENS_OFF, P)
    return {"R0": R0(theta[0], P=P), "reduction": 100 * (1 - c_pk / b_pk), "min_p": 100 * min_p_to_halve(theta, SENS_OFF, P)}

t0 = time.perf_counter()
N_BASE = N
ref = sensitivity_outputs(FIXED, K_NB)
sens = []
for name in ["sigma_h", "gamma", "sigma_v", "mu_v", "m0", "SUSC_FRAC", "k"]:
    out = {}
    for f in (0.75, 1.25):
        P = dict(FIXED); k = K_NB
        if name == "SUSC_FRAC":
            N = N_BASE * f                        # weekly_cases and the starting state read the global N
        elif name == "k":
            k = K_NB * f
        else:
            P[name] = FIXED[name] * f
        try:
            out[f] = sensitivity_outputs(P, k)
        finally:
            N = N_BASE
    sens.append({"input": name, **{f"{key} at -25%": out[0.75][key] for key in ref}, **{f"{key} at +25%": out[1.25][key] for key in ref}})
sens = pd.DataFrame(sens).set_index("input")
print(f"Reference (posterior mode): R0 = {ref['R0']:.2f}; campaign p = {SENS_P:.0%} starting {SENS_OFF} weeks before the peak "
      f"cuts the peak by {ref['reduction']:.1f}%; smallest p that halves the peak at {SENS_OFF} weeks = {ref['min_p']:.1f}%")
print(sens.round(2).to_string())
print(f"{time.perf_counter() - t0:.0f} s")

lab = {"sigma_h": "human incubation rate σh", "gamma": "recovery rate γ", "sigma_v": "mosquito incubation rate σv",
       "mu_v": "mosquito death rate μv", "m0": "mosquitoes per human m0", "SUSC_FRAC": "susceptible fraction",
       "k": "NB dispersion k"}
order = (sens["min_p at -25%"] - sens["min_p at +25%"]).abs().sort_values().index
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for j, name in enumerate(order):
    lo_v, hi_v = sens.loc[name, "min_p at -25%"], sens.loc[name, "min_p at +25%"]
    ax.barh(j, lo_v - ref["min_p"], left=ref["min_p"], color="tab:blue", alpha=0.8, label="input −25%" if j == 0 else None)
    ax.barh(j, hi_v - ref["min_p"], left=ref["min_p"], color="tab:orange", alpha=0.8, label="input +25%" if j == 0 else None)
ax.axvline(ref["min_p"], color="k", lw=1)
ax.set_yticks(range(len(order)), [lab[n] for n in order])
ax.set_xlabel(f"smallest cut p (%) that halves the peak, campaign starting {SENS_OFF} weeks before it")
ax.set_title("Sensitivity to fixed inputs (±25%, model refitted each time)"); ax.legend(loc="lower right", fontsize=8); fig.tight_layout()
fig.savefig("figures/tornado.png", dpi=150); plt.show()

### 5.6 Reference values for the web demo

In [ ]:
b_pk, c_pk, _, _ = campaign_effect(THETA_MED, SENS_P, SENS_OFF)
DEMO_REF = {"baseline_peak": round(float(b_pk), 1), "campaign_peak": round(float(c_pk), 1),
            "campaign_reduction_pct": round(float(100 * (1 - c_pk / b_pk)), 1)}
print("Web demo reference values (deterministic RK4, dt = 0.5 day, posterior-median theta):")
print(f"  baseline peak = {b_pk:.1f} reported cases per week (epi week {PEAK_WEEK})")
print(f"  with p = {SENS_P:.0%} starting {SENS_OFF} weeks before the peak: peak = {c_pk:.1f} ({DEMO_REF['campaign_reduction_pct']}% lower)")
print("Values to hard-code in docs/index.html:")
print({"beta": round(float(THETA_MED[0]), 6), "rho": round(float(THETA_MED[1]), 6), "I0": round(float(THETA_MED[2]), 3),
       "N": float(N), **{k: round(v, 6) for k, v in FIXED.items()}, "start_week": FIT_START, "horizon_weeks": HORIZON, "dt": DT})

## 6. Discussion and limitations

**Answer.** In the calibrated model, halving the peak weekly case count needs a campaign that starts about **8 weeks before the peak** and removes **at least half** of new adult mosquitoes: $p = 50\%$ gives a median 51% peak reduction (95% interval 48–53%), and $p = 60\%$ puts the whole interval above 50% (55–63%). Starting 6 weeks before the peak needs about 80% (51%, 45–58%). Starting 4 weeks or less before the peak never halves it within the grid: even 80% gives only 27% (22–34%).

**Why timing dominates.** A campaign only stops *new* mosquitoes. Adults already alive die off over about two weeks ($1/\mu_v = 14$ days), the virus then needs about 10 days inside the mosquito and about 5.5 days in the person. The result is a delay of 3–4 weeks before a campaign shows up in reported cases. A campaign launched once a surge is visible is too late for the peak, although it still lowers total cases (for example 8–14% fewer for 50–70% campaigns started 2 weeks before the peak). For Colombo in 2026, with the peak in early July (week 28), 8 weeks earlier is mid-May, right at the onset of the south-west monsoon and before the rise was obvious in the data.

**Robustness.** With each fixed input changed by ±25% and the model refitted, the smallest cut that halves the peak from 8 weeks stays between 45% and 59%. The mosquito death rate $\mu_v$ matters most, because it sets how quickly the adult population falls after a campaign starts. The susceptible fraction and the dispersion $k$ have no effect after refitting, confirming that the susceptible fraction only trades off with the reporting fraction $\rho$.

**Limitations**
1. One constant transmission rate: the flat pre-monsoon weeks 16–19 cannot be fitted, so the window starts at week 20, and campaigns earlier than 8 weeks before the peak cannot be assessed.
2. A special control week (15–21 June, week 25) lies inside the fitted window; its effect is absorbed into the baseline. Weeks 24–26 are missing from the data, so it could not be seen directly.
3. A campaign is a constant, lasting cut in mosquito recruitment; real campaigns vary in coverage and fade over time.
4. Homogeneous mixing across the whole district, one serotype, deterministic dynamics.
5. The fixed biological rates are literature values; the susceptible fraction (0.30) is assumed.

**Web demo reference value:** with its default settings (no campaign), the demo must reproduce a baseline peak of **1474.6 reported cases per week in epi week 28**. With $p = 50\%$ starting 8 weeks before the peak it must give **728.8** (50.6% lower). Both values are printed in section 5.6 and must match to within 1%.

**Run time:** see the output of the last cell (target under 10 minutes on a free Colab runtime).

## 7. References

[1] W. O. Kermack and A. G. McKendrick, "A contribution to the mathematical theory of epidemics," *Proc. R. Soc. Lond. A*, vol. 115, no. 772, pp. 700–721, 1927, doi: 10.1098/rspa.1927.0118.

[2] L. Esteva and C. Vargas, "Analysis of a dengue disease transmission model," *Math. Biosci.*, vol. 150, no. 2, pp. 131–151, 1998, doi: 10.1016/S0025-5564(98)10003-2.

[3] E. A. C. Newton and P. Reiter, "A model of the transmission of dengue fever with an evaluation of the impact of ultra-low volume (ULV) insecticide applications on dengue epidemics," *Am. J. Trop. Med. Hyg.*, vol. 47, no. 6, pp. 709–720, 1992, doi: 10.4269/ajtmh.1992.47.709.

[4] M. Andraud, N. Hens, C. Marais and P. Beutels, "Dynamic epidemiological models for dengue transmission: a systematic review of structural approaches," *PLoS ONE*, vol. 7, no. 11, e49085, 2012, doi: 10.1371/journal.pone.0049085.

[5] P. van den Driessche and J. Watmough, "Reproduction numbers and sub-threshold endemic equilibria for compartmental models of disease transmission," *Math. Biosci.*, vol. 180, no. 1–2, pp. 29–48, 2002, doi: 10.1016/S0025-5564(02)00108-6.

[6] N. Metropolis, A. W. Rosenbluth, M. N. Rosenbluth, A. H. Teller and E. Teller, "Equation of state calculations by fast computing machines," *J. Chem. Phys.*, vol. 21, no. 6, pp. 1087–1092, 1953, doi: 10.1063/1.1699114.

[7] G. O. Roberts, A. Gelman and W. R. Gilks, "Weak convergence and optimal scaling of random walk Metropolis algorithms," *Ann. Appl. Probab.*, vol. 7, no. 1, pp. 110–120, 1997, doi: 10.1214/aoap/1034625254.

[8] A. Gelman and D. B. Rubin, "Inference from iterative simulation using multiple sequences," *Statist. Sci.*, vol. 7, no. 4, pp. 457–472, 1992, doi: 10.1214/ss/1177011136.

[9] A. M. Law, *Simulation Modeling and Analysis*, 5th ed. New York, NY, USA: McGraw-Hill, 2015.

[10] Epidemiology Unit, Ministry of Health, Sri Lanka, *Weekly Epidemiological Report*, vol. 53, 2026. [Online]. Available: https://www.epid.gov.lk

[11] T. S. Talagala, *denguedatahub*, R package, GitHub repository, 2026. [Online]. Available: https://github.com/thiyangt/denguedatahub

[12] Department of Census and Statistics, Sri Lanka, *Census of Population and Housing 2024: Final Report*, 2026. [Online]. Available: https://www.statistics.gov.lk

In [ ]:
# Run-all time and figure bundle
import shutil
shutil.make_archive("figures", "zip", "figures")      # download figures.zip from the Files panel
print(f"Total run time: {(time.perf_counter() - T_START) / 60:.1f} minutes")
print("Figures saved:", sorted(os.listdir("figures")))